# Graph Intelligence: Neo4j Aura Agent + OpenAI Agents SDK

An **Aura Agent** is a managed agent that Neo4j hosts alongside your database. You configure it in the
Aura Console - its ontology, its tools, its instructions - and Neo4j runs it. It does its own
chain-of-thought reasoning over the graph and returns an answer rather than rows.

Exposed as an **MCP server**, it becomes callable from anywhere. This notebook connects one to the
[OpenAI Agents SDK](https://openai.github.io/openai-agents-python/) as a **sub-agent**: your agent
delegates graph questions to the hosted one and handles everything else itself.

1. **Machine-to-machine authentication** - a bearer token with no browser step
2. **MCP connection** - `MCPServerStreamableHttp` pointed at the hosted agent
3. **Agent as a tool** - wrapping the hosted agent so an orchestrator can call it alongside local tools

Nothing runs locally - the agent already runs in Aura.

## 1. Setup

Install the required Python packages.

In [1]:
!pip install --quiet --upgrade openai-agents requests
print("Packages installed ✓")


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Packages installed ✓


## 2. About the agent used here

This notebook connects to an **Investment Support Agent** built in the Neo4j Aura Console. It is just
an example - the integration below works with any Aura Agent, so use your own if you have one.

To create one:

1. In the [Aura Console](https://console.neo4j.io/), enable **GenAI assistance** for your organization
   and **Tool authentication** for your database.
2. Create an agent, point it at your instance, and configure its ontology, tools, and instructions.
3. Open **Configure → External access**, enable the **MCP server**, and update the agent.
4. Copy the MCP endpoint from the agent's `[…]` menu.

The [getting started tutorial](https://neo4j.com/developer/genai-ecosystem/aura-agent-getting-started/)
walks through it end to end.

Because all of the graph reasoning lives in the agent's own configuration, nothing below is specific
to this agent - only the questions in the demo cells assume investment data. Swap those for whatever
your agent knows about.

## 3. Configuration

Two credentials and one URL, all from the Aura Console.

**Client credentials.** Profile menu → **Account settings** → **Client credentials** tab →
**Aura Agent & MCP** → **Create client credential**. Save the generated values, they are shown once.

> These are *not* Aura API keys. API keys authenticate against `api.neo4j.io` and work with the
> agent's REST endpoint; the MCP endpoint uses client credentials from the Aura Agent & MCP tab and a
> different token endpoint.

**MCP endpoint URL**, copied from the agent's `[…]` menu:

```
https://mcp.neo4j.io/agent?project_id=<PROJECT_ID>&agent_id=<AGENT_ID>
```

Set everything as environment variables so nothing sensitive lands in the notebook:

```bash
export AURA_MCP_CLIENT_ID="..."
export AURA_MCP_CLIENT_SECRET="..."
export AURA_AGENT_MCP_URL="https://mcp.neo4j.io/agent?project_id=...&agent_id=..."
```

> **Cost:** An externally accessible agent incurs charges per [Neo4j Aura pricing](https://neo4j.com/pricing/).
> Availability can be switched off again from the same menu.

In [ ]:
import asyncio
import json
import os
import time
from getpass import getpass

import requests

from agents import Agent, Runner, function_tool
from agents.mcp import MCPServerStreamableHttp

from dotenv import load_dotenv
load_dotenv()
# OpenAI API key
os.environ.setdefault("OPENAI_API_KEY", getpass("OpenAI API key: "))

MODEL = "gpt-5.4-mini"

# Gateway token endpoint for the agent MCP server, and the fixed audience it issues for.
AURA_MCP_TOKEN_URL = "https://mcp.neo4j.io/oauth/token"
AURA_MCP_AUDIENCE = "https://agent-mcp.neo4j.io"

CLIENT_ID = os.environ.get("AURA_MCP_CLIENT_ID") or getpass("Aura MCP client ID: ")
CLIENT_SECRET = os.environ.get("AURA_MCP_CLIENT_SECRET") or getpass("Aura MCP client secret: ")
AGENT_MCP_URL = os.environ.get("AURA_AGENT_MCP_URL") or input("Agent MCP endpoint URL: ").strip()

# Show the endpoint without leaking the IDs into notebook output.
print("Configuration set.")
print("Endpoint:", AGENT_MCP_URL.split("?")[0], "(project and agent IDs hidden)")
print("Model:   ", MODEL)

Configuration set.
Endpoint:  (project and agent IDs hidden)
Model:    gpt-5.4-mini


## 4. Get a token

The agent MCP server supports two authorization flows: **user (authorization code)**, which is the
browser login an MCP client like Claude Desktop or VS Code performs, and **machine-to-machine (client
credentials)**, for scripts and services. We use the second - no browser, so this runs from a backend
or a scheduled job.

Post a `client_credentials` grant to the gateway token endpoint with the fixed audience
`https://agent-mcp.neo4j.io`, then send the returned JWT as a bearer token.

> **The token endpoint allows 15 requests per hour per client ID.** Cache the token for its full
> `expires_in` window rather than fetching one per request. The helper below does that, so re-running
> cells reuses the cached token instead of spending quota.

In [ ]:
_token = {"value": None, "expires_at": 0}


def get_token(force: bool = False) -> str:
    """Return a cached bearer token, fetching a new one only when it is near expiry.

    The gateway allows 15 token requests per hour per client ID, so caching is required
    rather than merely efficient.
    """
    if not force and _token["value"] and time.time() < _token["expires_at"] - 60:
        return _token["value"]

    response = requests.post(
        AURA_MCP_TOKEN_URL,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "audience": AURA_MCP_AUDIENCE,
        },
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    _token["value"] = payload["access_token"]
    _token["expires_at"] = time.time() + payload.get("expires_in", 3600)
    return _token["value"]


get_token()
print(f"Token cached, valid for {int(_token['expires_at'] - time.time())}s ✓")

## 5. Check the endpoint

Call the MCP endpoint directly before handing it to a model. If the credentials or URL are wrong, the
error is obvious here; inside an agent transcript it is not.

This also shows which tools the agent exposes, which is what the model will see.

In [ ]:
response = requests.post(
    AGENT_MCP_URL,
    headers={
        "Authorization": f"Bearer {get_token()}",
        "Content-Type": "application/json",
        # Streamable HTTP can answer as JSON or as a server-sent event stream.
        "Accept": "application/json, text/event-stream",
    },
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
    timeout=60,
)
response.raise_for_status()

body = response.text
if "text/event-stream" in response.headers.get("Content-Type", ""):
    body = next(line[5:] for line in body.splitlines() if line.startswith("data:"))

for t in json.loads(body)["result"]["tools"]:
    print(f"  {t['name']}")
    print(f"    {t.get('description', '').splitlines()[0][:100]}")

## 6. Connect it to the Agents SDK

`MCPServerStreamableHttp` connects to the remote endpoint and exposes its tools to an agent. The
bearer token goes in the headers.

Two settings worth noting. Hosted agents plan and query before replying, so the timeouts are longer
than you would use for a database call. And `cache_tools_list=True` avoids re-listing the agent's
tools on every run - safe here because a hosted agent's tool set only changes when you reconfigure it
in the Console.

In [ ]:
aura_mcp = MCPServerStreamableHttp(
    params={
        "url": AGENT_MCP_URL,
        "headers": {"Authorization": f"Bearer {get_token()}"},
        "timeout": 120,          # hosted agents reason before answering
    },
    name="aura-agent",
    cache_tools_list=True,
    client_session_timeout_seconds=120,
)

# Connect once - the session stays alive for all subsequent cells.
await aura_mcp.connect()
tools = await aura_mcp.list_tools()

print("Connected to the Aura Agent MCP ✓")
print("Available tools:", [t.name for t in tools])

Note what the instructions below do *not* contain: no schema, no Cypher guidance, no description of
the graph. All of that lives in the Aura Agent's configuration. This agent's job is to decide *when*
to consult it and how to present the answer.

In [ ]:
GRAPH_PROMPT = """You help users with investment research questions.

A hosted Neo4j Aura Agent has access to the investment knowledge graph - use its tools for anything
about companies, holdings, or relationships in that data. Pass the user's question through clearly
rather than rewriting it into keywords.

Present what comes back in your own words, and say plainly if the agent could not answer rather than
filling the gap from your own knowledge."""


async def run_graph_agent(query: str):
    agent = Agent(
        name="investment_assistant",
        instructions=GRAPH_PROMPT,
        mcp_servers=[aura_mcp],
        model=MODEL,
    )
    print(f"Query: {query}\n")
    result = await Runner.run(agent, query)
    print(f"Result: {result.final_output}")
    return result


await run_graph_agent("What can you tell me about the companies in this knowledge graph?")

The tool call goes out to Aura, where the hosted agent plans, runs its own queries, and reasons over
the results. What returns is an answer, not rows - your agent then decides how to present it.

That division is the point of this integration. The graph logic lives with the graph and is maintained
in the Console; the conversational layer lives in your application. Changing the agent's ontology or
tools takes effect without touching this notebook.

## 7. The hosted agent as a sub-agent

The Agents SDK lets an agent be used as a tool. `as_tool()` wraps one agent so another can call it,
which fits a hosted specialist exactly: the orchestrator keeps control of the conversation and calls
the graph specialist when a question needs it.

Everything else - your own APIs, calculations, internal services - stays in your application as
ordinary `@function_tool` definitions. The model chooses between them.

In [ ]:
@function_tool
def portfolio_weight(position_value: float, portfolio_total: float) -> str:
    """Calculate what percentage of a portfolio a single position represents."""
    if portfolio_total <= 0:
        return "Portfolio total must be greater than zero."
    return f"{(position_value / portfolio_total) * 100:.2f}%"


# The hosted Aura Agent, wrapped as a local specialist.
graph_specialist = Agent(
    name="graph_specialist",
    instructions=(
        "You answer questions about companies, holdings, and their relationships using the "
        "hosted Neo4j Aura Agent's tools. Pass the question through clearly and report what "
        "comes back. Do not add outside knowledge."
    ),
    mcp_servers=[aura_mcp],
    model=MODEL,
)

orchestrator = Agent(
    name="research_assistant",
    instructions=(
        "You help users with investment research.\n"
        "Use `ask_knowledge_graph` for anything about companies, holdings, or relationships "
        "in the graph. Use the local tools for calculations. Combine the results into one "
        "clear answer."
    ),
    tools=[
        graph_specialist.as_tool(
            tool_name="ask_knowledge_graph",
            tool_description=(
                "Ask the hosted Neo4j knowledge graph agent a question about companies, "
                "holdings, sectors, or relationships between organizations."
            ),
        ),
        portfolio_weight,
    ],
    model=MODEL,
)

print("Orchestrator with graph sub-agent and 1 local tool ✓")

In [ ]:
query = (
    "If I hold 45000 of a company in a 750000 portfolio, what weight is that? "
    "And what does the knowledge graph say about that company's sector?"
)
print(f"Query: {query}\n")

result = await Runner.run(orchestrator, query)
print(f"Result: {result.final_output}")

print("\nTool calls made:")
for item in result.new_items:
    name = getattr(getattr(item, "raw_item", None), "name", None)
    if name:
        print(f"  - {name}")

The calculation runs locally and the graph question goes to Aura. Neither side needs to know about the
other, and the orchestrator keeps control of the conversation throughout - unlike a handoff, which
would transfer the rest of the turn to the specialist.

## 8. Token expiry

`MCPServerStreamableHttp` captures its headers when constructed, so the token it holds does not refresh
itself. For a notebook that rarely matters. For a long-running service it does - once the token
expires, calls start failing with 401.

The fix is to rebuild the server connection with a fresh token. `get_token()` returns the cached value
until it is close to expiring, so calling this repeatedly does not burn the hourly token quota.

In [ ]:
async def connect_aura_mcp() -> MCPServerStreamableHttp:
    """Open an MCP connection with a current token. Call again after a 401."""
    server = MCPServerStreamableHttp(
        params={
            "url": AGENT_MCP_URL,
            "headers": {"Authorization": f"Bearer {get_token()}"},
            "timeout": 120,
        },
        name="aura-agent",
        cache_tools_list=True,
        client_session_timeout_seconds=120,
    )
    await server.connect()
    return server


# An alternative for production: front the endpoint with a small local proxy that
# injects a current token on every request. The MCP client then talks to a stable
# URL and never sees a credential.
print("Reconnect helper defined ✓")

In [ ]:
await aura_mcp.cleanup()
print("MCP session closed ✓")

## 9. Summary

| Pattern | Key APIs | Use Case |
|---------|----------|----------|
| Hosted agent over MCP | `MCPServerStreamableHttp`, `Agent(mcp_servers=[...])` | Delegate graph reasoning to an agent Neo4j hosts |
| Agent as a tool | `Agent.as_tool()` | An orchestrator calls the graph specialist and keeps control of the conversation |
| Machine-to-machine auth | `client_credentials` grant, cached bearer token | Headless access from a backend or scheduled job |

### Key Implementation Notes

- **Credentials** - client credentials come from **Account settings → Client credentials → Aura Agent & MCP**. Aura API keys are a different credential for `api.neo4j.io`, and belong to the agent's REST endpoint rather than its MCP endpoint.
- **Token endpoint** - `https://mcp.neo4j.io/oauth/token` with audience `https://agent-mcp.neo4j.io`, rate-limited to 15 requests per hour per client ID. Cache each token for its full lifetime.
- **Timeouts** - a hosted agent plans and queries before replying, so allow more time than a database call would need.
- **Token expiry** - `MCPServerStreamableHttp` fixes its headers at construction. Rebuild the connection on refresh, or front the endpoint with a proxy that injects a current token per request.
- **Graph logic stays with the graph** - the instructions carry no schema and no Cypher, so updating the agent's ontology in the Console changes behaviour without a code change here.

### Resources

- [OpenAI Agents SDK Documentation](https://openai.github.io/openai-agents-python/)
- [OpenAI Agents SDK - MCP Support](https://openai.github.io/openai-agents-python/mcp/)
- [OpenAI Agents SDK - Agents as tools](https://openai.github.io/openai-agents-python/tools/)
- [Neo4j Aura Agent](https://neo4j.com/docs/aura/aura-agent/)
- [Aura Agent getting started](https://neo4j.com/developer/genai-ecosystem/aura-agent-getting-started/)